In [ ]:
import json, os

os.makedirs("/kaggle/working/data/mock", exist_ok=True)

job_skills = [
    {"esco_id": "skill_sql",     "label": "SQL",                "importance": 0.9},
    {"esco_id": "skill_viz",     "label": "Data visualisation", "importance": 0.8},
    {"esco_id": "skill_python",  "label": "Python",             "importance": 0.7},
    {"esco_id": "skill_stats",   "label": "Statistics",         "importance": 0.6},
    {"esco_id": "skill_excel",   "label": "Excel",              "importance": 0.4},
]

with open("/kaggle/working/data/mock/job_skills.json", "w") as f:
    json.dump(job_skills, f, indent=2)

print("Saved", len(job_skills), "skills")

In [ ]:
# ------------------------------------------------------------
# Mock learner estimates  (stand-in for Member 4's output)
# Shape agreed by the team:  { esco_id : mastery }
#   esco_id : same skill code used in job_skills.json
#   mastery : 0 to 1  ->  0 = knows nothing, 1 = knows it fully
# Later this file will be replaced by the real learner model.
# ------------------------------------------------------------
import json

learner_estimates = {
    "skill_sql":    0.2,   # weak: this should become the biggest gap
    "skill_viz":    0.8,   # strong
    "skill_python": 0.5,   # medium
    "skill_stats":  0.4,   # medium-low
    "skill_excel":  0.9,   # very strong
}

with open("/kaggle/working/data/mock/learner_estimates.json", "w") as f:
    json.dump(learner_estimates, f, indent=2)

print("Saved", len(learner_estimates), "skills")

In [3]:
# ------------------------------------------------------------
# Mock course catalogue  (stand-in for the real course list)
# Columns agreed by the team:
#   course_id        : unique code for the course
#   title            : course name
#   description      : one line about the course
#   esco_skill_ids   : skills the course teaches (separated by |)
#   difficulty       : beginner / intermediate / advanced
#   estimated_hours  : how long the course takes
#   prerequisites    : course_id that should be done first (empty = none)
#   resource_url     : where to find the course (fake for now)
# ------------------------------------------------------------
import pandas as pd

courses = [
    ["c01", "SQL Basics",                "Learn to write simple SQL queries",       "skill_sql",                   "beginner",     6,  "",    "https://example.com/c01"],
    ["c02", "Advanced SQL",              "Joins, subqueries and window functions",  "skill_sql",                   "advanced",     12, "c01", "https://example.com/c02"],
    ["c03", "Python for Beginners",      "First steps with Python programming",     "skill_python",                "beginner",     10, "",    "https://example.com/c03"],
    ["c04", "Data Visualisation 101",    "Build charts and dashboards",             "skill_viz",                   "beginner",     8,  "",    "https://example.com/c04"],
    ["c05", "Statistics Foundations",    "Averages, probability and distributions", "skill_stats",                 "beginner",     14, "",    "https://example.com/c05"],
    ["c06", "Excel Essentials",          "Formulas, tables and pivot tables",       "skill_excel",                 "beginner",     5,  "",    "https://example.com/c06"],
    ["c07", "Data Analysis with Python", "Analyse data using pandas and charts",    "skill_python|skill_viz",      "intermediate", 20, "c03", "https://example.com/c07"],
    ["c08", "Data Analyst Bootcamp",     "SQL, Python and statistics in one course","skill_sql|skill_python|skill_stats", "intermediate", 40, "",    "https://example.com/c08"],
]

columns = ["course_id", "title", "description", "esco_skill_ids",
           "difficulty", "estimated_hours", "prerequisites", "resource_url"]

courses_df = pd.DataFrame(courses, columns=columns)
courses_df.to_csv("/kaggle/working/data/mock/courses.csv", index=False)

courses_df

,course_id,title,description,esco_skill_ids,difficulty,estimated_hours,prerequisites,resource_url
0,c01,SQL Basics,Learn to write simple SQL queries,skill_sql,beginner,6,,https://example.com/c01
1,c02,Advanced SQL,"Joins, subqueries and window functions",skill_sql,advanced,12,c01,https://example.com/c02
2,c03,Python for Beginners,First steps with Python programming,skill_python,beginner,10,,https://example.com/c03
3,c04,Data Visualisation 101,Build charts and dashboards,skill_viz,beginner,8,,https://example.com/c04
4,c05,Statistics Foundations,"Averages, probability and distributions",skill_stats,beginner,14,,https://example.com/c05
5,c06,Excel Essentials,"Formulas, tables and pivot tables",skill_excel,beginner,5,,https://example.com/c06
6,c07,Data Analysis with Python,Analyse data using pandas and charts,skill_python|skill_viz,intermediate,20,c03,https://example.com/c07
7,c08,Data Analyst Bootcamp,"SQL, Python and statistics in one course",skill_sql|skill_python|skill_stats,intermediate,40,,https://example.com/c08


In [4]:
# ------------------------------------------------------------
# Load the three input files and check that skill IDs match
# ------------------------------------------------------------
import json
import pandas as pd

BASE = "/kaggle/working/data/mock"

with open(f"{BASE}/job_skills.json") as f:
    job_skills = json.load(f)

with open(f"{BASE}/learner_estimates.json") as f:
    learner_estimates = json.load(f)

courses_df = pd.read_csv(BASE + "/courses.csv", keep_default_na=False)

# Collect the skill IDs from each file
job_ids     = {s["esco_id"] for s in job_skills}
learner_ids = set(learner_estimates.keys())
course_ids  = {sid for cell in courses_df["esco_skill_ids"] for sid in cell.split("|")}

# Check 1: every job skill has a learner mastery value
print("Job skills missing from learner file:", job_ids - learner_ids)

# Check 2: every job skill is taught by at least one course
print("Job skills no course teaches:        ", job_ids - course_ids)

# Check 3: no course teaches a skill unknown to the job file (just informational)
print("Course skills not needed by the job: ", course_ids - job_ids)

Job skills missing from learner file: set()
Job skills no course teaches:         set()
Course skills not needed by the job:  set()


In [ ]:
# ------------------------------------------------------------
# Skill gap:  Priority(s) = importance * (1 - mastery)
#   importance : how much the job needs the skill (0 to 1)
#   mastery    : how well the learner knows it   (0 to 1)
# Big priority = important skill AND the learner is weak at it
# ------------------------------------------------------------
def compute_skill_gaps(job_skills, learner_estimates):
    gaps = []
    for s in job_skills:
        d = s["importance"]
        m = learner_estimates.get(s["esco_id"], 0.0)   # unknown skill -> assume 0
        gaps.append({
            "esco_id":  s["esco_id"],
            "label":    s["label"],
            "importance": d,
            "mastery":  m,
            "priority": round(d * (1 - m), 3),
        })
    return sorted(gaps, key=lambda g: g["priority"], reverse=True)

gaps = compute_skill_gaps(job_skills, learner_estimates)

pd.DataFrame(gaps)

,esco_id,label,importance,mastery,priority
0,skill_sql,SQL,0.9,0.2,0.72
1,skill_stats,Statistics,0.6,0.4,0.36
2,skill_python,Python,0.7,0.5,0.35
3,skill_viz,Data visualisation,0.8,0.8,0.16
4,skill_excel,Excel,0.4,0.9,0.04


In [ ]:
# ------------------------------------------------------------
# Course score:  Score(c) = sum(Priority of skills it teaches) / hours
# Higher score = more gap closed per hour of study
# ------------------------------------------------------------
def score_courses(gaps, courses_df):
    priority = {g["esco_id"]: g["priority"] for g in gaps}   # skill -> priority
    rows = []
    for _, c in courses_df.iterrows():
        skills = c["esco_skill_ids"].split("|")
        total = sum(priority.get(s, 0.0) for s in skills)    # skills not in the gap list add 0
        rows.append({
            "course_id": c["course_id"],
            "title": c["title"],
            "hours": c["estimated_hours"],
            "gap_closed": round(total, 3),
            "score": round(total / c["estimated_hours"], 4),
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False)

scores = score_courses(gaps, courses_df)
scores

,course_id,title,hours,gap_closed,score
0,c01,SQL Basics,6,0.72,0.1200
1,c02,Advanced SQL,12,0.72,0.0600
7,c08,Data Analyst Bootcamp,40,1.43,0.0357
2,c03,Python for Beginners,10,0.35,0.0350
4,c05,Statistics Foundations,14,0.36,0.0257
6,c07,Data Analysis with Python,20,0.51,0.0255
3,c04,Data Visualisation 101,8,0.16,0.0200
5,c06,Excel Essentials,5,0.04,0.0080


In [ ]:
# ------------------------------------------------------------
# Prerequisite filter
#   completed : set of course_ids the learner has already finished
#   A course is kept only if all its prerequisites are completed
#   (empty prerequisite = no requirement, always kept)
# ------------------------------------------------------------
def filter_by_prerequisites(scores, courses_df, completed=None):
    completed = completed or set()
    prereq = dict(zip(courses_df["course_id"], courses_df["prerequisites"]))

    def unlocked(course_id):
        needed = [p for p in prereq[course_id].split("|") if p]
        return all(p in completed for p in needed)

    return scores[scores["course_id"].apply(unlocked)]

# Ayesha has completed nothing yet
available = filter_by_prerequisites(scores, courses_df, completed=set())
available

,course_id,title,hours,gap_closed,score
0,c01,SQL Basics,6,0.72,0.1200
7,c08,Data Analyst Bootcamp,40,1.43,0.0357
2,c03,Python for Beginners,10,0.35,0.0350
4,c05,Statistics Foundations,14,0.36,0.0257
3,c04,Data Visualisation 101,8,0.16,0.0200
5,c06,Excel Essentials,5,0.04,0.0080


In [ ]:
# ------------------------------------------------------------
# recommend_courses: the one function the team agreed on
#   Input : job_skills, learner_estimates, course_catalogue
#   Output: [{course_id, score, reason}]  (top N courses)
# Steps inside: gaps -> score courses -> remove locked -> top N
# ------------------------------------------------------------
import json

def recommend_courses(job_skills, learner_estimates, course_catalogue,
                      completed=None, top_n=5):
    gaps = compute_skill_gaps(job_skills, learner_estimates)
    gaps_by_id = {g["esco_id"]: g for g in gaps}

    scores = score_courses(gaps, course_catalogue)
    available = filter_by_prerequisites(scores, course_catalogue, completed)
    top = available.head(top_n)

    skills_of = dict(zip(course_catalogue["course_id"],
                         course_catalogue["esco_skill_ids"]))
    results = []
    for _, row in top.iterrows():
        parts = [f"{gaps_by_id[s]['label']} (priority {gaps_by_id[s]['priority']})"
                 for s in skills_of[row["course_id"]].split("|")
                 if s in gaps_by_id and gaps_by_id[s]["priority"] > 0]
        results.append({
            "course_id": row["course_id"],
            "score": float(row["score"]),
            "reason": f"Teaches {', '.join(parts)} in {int(row['hours'])} hours",
        })
    return results

recs = recommend_courses(job_skills, learner_estimates, courses_df)

with open("/kaggle/working/data/mock/recommendations.json", "w") as f:
    json.dump(recs, f, indent=2)

for r in recs:
    print(r)

{'course_id': 'c01', 'score': 0.12, 'reason': 'Teaches SQL (priority 0.72) in 6 hours'}
{'course_id': 'c08', 'score': 0.0357, 'reason': 'Teaches SQL (priority 0.72), Python (priority 0.35), Statistics (priority 0.36) in 40 hours'}
{'course_id': 'c03', 'score': 0.035, 'reason': 'Teaches Python (priority 0.35) in 10 hours'}
{'course_id': 'c05', 'score': 0.0257, 'reason': 'Teaches Statistics (priority 0.36) in 14 hours'}
{'course_id': 'c04', 'score': 0.02, 'reason': 'Teaches Data visualisation (priority 0.16) in 8 hours'}


In [ ]:
from datasets import load_dataset

df = load_dataset("azrai99/coursera-course-dataset")
import pandas as pd

# df is currently a DatasetDict; take the "train" split and convert to pandas
df = df["train"].to_pandas()

print("Rows:", len(df))
print()
print("Schedule values:")
print(df["Schedule"].value_counts().head(15))
print()
print("Level values:")
print(df["Level"].value_counts())
print()
print("Sample of Skills:")
print(df["Skills"].head(5).tolist())

Rows: 6645

Schedule values:
Schedule
1 month (at 10 hours a week)                        483
9 hours to complete (3 weeks at 3 hours a week)     248
7 hours to complete (3 weeks at 2 hours a week)     241
6 hours to complete (3 weeks at 2 hours a week)     228
2 months (at 10 hours a week)                       216
5 hours to complete (3 weeks at 1 hour a week)      214
10 hours to complete (3 weeks at 3 hours a week)    213
8 hours to complete (3 weeks at 2 hours a week)     207
13 hours to complete (3 weeks at 4 hours a week)    193
11 hours to complete (3 weeks at 3 hours a week)    190
12 hours to complete (3 weeks at 4 hours a week)    173
1 hour to complete                                  166
4 hours to complete (3 weeks at 1 hour a week)      160
2 hours to complete                                 143
3 hours to complete (3 weeks at 1 hour a week)      141
Name: count, dtype: int64

Level values:
Level
Beginner level        3591
Intermediate level    2024
Advanced level       

In [ ]:
# ------------------------------------------------------------
# Convert Schedule text -> estimated_hours (a number)
#   "9 hours to complete (...)"      -> 9
#   "1 month (at 10 hours a week)"   -> 1 * 4 weeks * 10 = 40
#   "3 weeks (at 5 hours a week)"    -> 3 * 5 = 15
# Anything we cannot read becomes None and is checked below
# ------------------------------------------------------------
import re

def schedule_to_hours(text):
    if not isinstance(text, str):
        return None
    t = text.lower()

    # Pattern 1: "N hours to complete"
    m = re.search(r"(\d+(?:\.\d+)?)\s*hours?\s*to complete", t)
    if m:
        return float(m.group(1))

    # Pattern 2: "N months/weeks (at H hours a week)"
    m = re.search(r"(\d+(?:\.\d+)?)\s*(month|week)s?\s*\(at\s*(\d+(?:\.\d+)?)\s*hours?", t)
    if m:
        n, unit, per_week = float(m.group(1)), m.group(2), float(m.group(3))
        weeks = n * 4 if unit == "month" else n
        return weeks * per_week

    return None

df["estimated_hours"] = df["Schedule"].apply(schedule_to_hours)

print("Converted:", df["estimated_hours"].notna().sum(), "of", len(df))
print()
print("Examples we could NOT convert:")
print(df.loc[df["estimated_hours"].isna(), "Schedule"].value_counts().head(10))
print()
print(df["estimated_hours"].describe())

Converted: 4757 of 6645

Examples we could NOT convert:
Series([], Name: count, dtype: int64)

count    4757.000000
mean       23.944713
std        32.846978
min         1.000000
25%         7.000000
50%        13.000000
75%        27.000000
max       480.000000
Name: estimated_hours, dtype: float64


In [ ]:
# ------------------------------------------------------------
# Why did 1888 courses fail?  Are their Schedule values missing?
# ------------------------------------------------------------
failed = df[df["estimated_hours"].isna()]

print("Failed courses:            ", len(failed))
print("...with missing Schedule:  ", failed["Schedule"].isna().sum())
print("...with a Schedule we missed:", failed["Schedule"].notna().sum())
print()

# Look at the very long courses too (max was 480 hours)
print("Longest courses:")
print(df.nlargest(5, "estimated_hours")[["title", "Schedule", "estimated_hours"]])

Failed courses:             1888
...with missing Schedule:   1888
...with a Schedule we missed: 0

Longest courses:
                                                  title  \
461   IBM Back-End Development Professional Certificate   
4428  Neuroscience of Leadership: Leading with Your ...   
4619  Microsoft Full-Stack Developer Professional Ce...   
1456  Tableau Business Intelligence Analyst  Profess...   
3733  Data Warehousing for Business Intelligence Spe...   

                            Schedule  estimated_hours  
461   12 months (at 10 hours a week)            480.0  
4428   15 months (at 7 hours a week)            420.0  
4619   9 months (at 10 hours a week)            360.0  
1456   8 months (at 10 hours a week)            320.0  
3733   8 months (at 10 hours a week)            320.0  


In [ ]:
# ------------------------------------------------------------
# 1) Parse Skills text like "['SQL', 'Python']" into a real list
# 2) Remove repeats (ignoring upper/lower case) and empty values
# 3) Keep only courses that have BOTH hours and at least 1 skill
# ------------------------------------------------------------
import ast

def clean_skills(text):
    try:
        items = ast.literal_eval(text) if isinstance(text, str) else []
    except (ValueError, SyntaxError):
        return []
    seen, result = set(), []
    for s in items:
        s = str(s).strip().lower().replace("|", " ")   # "|" is our separator, so never keep it inside a skill
        if s and s not in seen:
            seen.add(s)
            result.append(s)
    return result

df["skill_list"] = df["Skills"].apply(clean_skills)

print("Courses before cleaning:     ", len(df))
print("...with no hours:            ", df["estimated_hours"].isna().sum())
print("...with no skills:           ", (df["skill_list"].str.len() == 0).sum())

clean_df = df[df["estimated_hours"].notna() & (df["skill_list"].str.len() > 0)].copy()

print("Courses kept:                ", len(clean_df))
print()
print("Skills per course:")
print(clean_df["skill_list"].str.len().describe())
print()
print("Sample:")
print(clean_df[["title", "estimated_hours", "skill_list"]].head(3).to_string())

Courses before cleaning:      6645
...with no hours:             1888
...with no skills:            1954
Courses kept:                 3454

Skills per course:
count    3454.000000
mean        4.493920
std         1.635241
min         1.000000
25%         4.000000
50%         5.000000
75%         5.000000
max        14.000000
Name: skill_list, dtype: float64

Sample:
                                       title  estimated_hours                                                                                                                                                                                     skill_list
2    JavaScript for Beginners Specialization             80.0                                                                                                                                      [web interactivty, jquery, data manipulation, javascript]
4          Understanding Fitness Programming             27.0                                                               

In [ ]:
# ------------------------------------------------------------
# For each job skill, count courses whose skill_list mentions it
# (simple text match; "sql" also catches "mysql", "postgresql")
# ------------------------------------------------------------
keywords = {
    "SQL":                "sql",
    "Data visualisation": "visuali",
    "Python":             "python",
    "Statistics":         "statistic",
    "Excel":              "excel",
}

for label, kw in keywords.items():
    n = clean_df["skill_list"].apply(lambda skills: any(kw in s for s in skills)).sum()
    print(f"{label:20s} {n} courses")

SQL                  78 courses
Data visualisation   127 courses
Python               161 courses
Statistics           112 courses
Excel                41 courses


In [ ]:
# ------------------------------------------------------------
# Build the real course catalogue in the team format
#   esco_skill_ids : our placeholder IDs, found by keyword match
#   difficulty     : "Beginner level" -> "beginner"
#   prerequisites  : empty (the dataset has none)
# Saved as courses_real.csv so the mock file stays untouched
# ------------------------------------------------------------
import re
import pandas as pd

rules = {
    "skill_sql":    r"\bsql\b|mysql|postgresql",
    "skill_viz":    r"visuali",
    "skill_python": r"\bpython\b",
    "skill_stats":  r"statistic",
    "skill_excel":  r"\bexcel\b",
}

def tag_skills(skill_list):
    found = [sid for sid, pattern in rules.items()
             if any(re.search(pattern, s) for s in skill_list)]
    return "|".join(found)

clean_df["esco_skill_ids"] = clean_df["skill_list"].apply(tag_skills)

real = clean_df[clean_df["esco_skill_ids"] != ""].copy()
real = real.reset_index(drop=True)

courses_real = pd.DataFrame({
    "course_id":       [f"r{i:04d}" for i in range(len(real))],
    "title":           real["title"],
    "description":     real["Description"].fillna("").str.slice(0, 200),
    "esco_skill_ids":  real["esco_skill_ids"],
    "difficulty":      real["Level"].str.replace(" level", "", regex=False).str.lower(),
    "estimated_hours": real["estimated_hours"],
    "prerequisites":   "",
    "resource_url":    real["URL"].fillna(""),
})

courses_real.to_csv("/kaggle/working/data/mock/courses_real.csv", index=False)

print("Courses saved:", len(courses_real))
print()
print("Courses per skill:")
for sid in rules:
    print(f"  {sid:14s}", courses_real["esco_skill_ids"].str.contains(sid).sum())
print()
print(courses_real[["course_id", "title", "esco_skill_ids", "estimated_hours"]].head(5).to_string())

Courses saved: 437

Courses per skill:
  skill_sql      72
  skill_viz      127
  skill_python   160
  skill_stats    112
  skill_excel    39

  course_id                                                 title          esco_skill_ids  estimated_hours
0     r0000                  How to Create and Explore a Data Set             skill_stats              3.0
1     r0001       Meta Database Engineer Professional Certificate  skill_sql|skill_python            144.0
2     r0002  Introduction to Machine Learning in Sports Analytics            skill_python             12.0
3     r0003                                Data Science Companion            skill_python              1.0
4     r0004              Python Programming for Quantum Computing            skill_python              7.0


In [ ]:
# ------------------------------------------------------------
# Run recommend_courses on the REAL catalogue
# Same function as Step 9, only the course file is different
# ------------------------------------------------------------
import pandas as pd

real_catalogue = pd.read_csv("/kaggle/working/data/mock/courses_real.csv",
                             keep_default_na=False)

recs_real = recommend_courses(job_skills, learner_estimates, real_catalogue, top_n=10)

# Add title, hours and difficulty so the output is readable
info = real_catalogue.set_index("course_id")[["title", "estimated_hours", "difficulty"]]

for r in recs_real:
    row = info.loc[r["course_id"]]
    print(f"{r['course_id']}  score={r['score']:.3f}  "
          f"{row['title']} ({row['estimated_hours']:.0f} h, {row['difficulty']})")
    print(f"        {r['reason']}")

r0255  score=0.520  Data Science for Professionals (1 h, beginner)
        Teaches Data visualisation (priority 0.16), Statistics (priority 0.36) in 1 hours
r0377  score=0.360  From Data to Decisions: Making Predictions with AI (1 h, intermediate)
        Teaches Statistics (priority 0.36) in 1 hours
r0291  score=0.350  The Fundamentals of Computing Capstone Exam (1 h, intermediate)
        Teaches Python (priority 0.35) in 1 hours
r0003  score=0.350  Data Science Companion (1 h, beginner)
        Teaches Python (priority 0.35) in 1 hours
r0196  score=0.350  Learn Python: 1 (1 h, beginner)
        Teaches Python (priority 0.35) in 1 hours
r0409  score=0.293  Power BI Data Analyst Associate Prep (3 h, intermediate)
        Teaches SQL (priority 0.72), Data visualisation (priority 0.16) in 3 hours
r0153  score=0.240  MySQL for Data Analytics and Business Intelligence (3 h, beginner)
        Teaches SQL (priority 0.72) in 3 hours
r0263  score=0.180  The Nature of Data and Relational Datab

In [ ]:
# ------------------------------------------------------------
# score_courses v2:  Score(c) = sum(Priority) / max(hours, MIN_HOURS)
# MIN_HOURS stops 1-2 hour courses from winning just because
# they are short. "hours" in the output is still the real value.
# ------------------------------------------------------------
MIN_HOURS = 5

def score_courses(gaps, courses_df, min_hours=MIN_HOURS):
    priority = {g["esco_id"]: g["priority"] for g in gaps}
    rows = []
    for _, c in courses_df.iterrows():
        skills = c["esco_skill_ids"].split("|")
        total = sum(priority.get(s, 0.0) for s in skills)
        rows.append({
            "course_id": c["course_id"],
            "title": c["title"],
            "hours": c["estimated_hours"],
            "gap_closed": round(total, 3),
            "score": round(total / max(c["estimated_hours"], min_hours), 4),
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False)

# Re-run the same recommendation as Step 15
recs_real = recommend_courses(job_skills, learner_estimates, real_catalogue, top_n=10)

for r in recs_real:
    row = info.loc[r["course_id"]]
    print(f"{r['course_id']}  score={r['score']:.3f}  "
          f"{row['title']} ({row['estimated_hours']:.0f} h, {row['difficulty']})")
    print(f"        {r['reason']}")

r0263  score=0.180  The Nature of Data and Relational Database Design (6 h, beginner)
        Teaches SQL (priority 0.72), Statistics (priority 0.36) in 6 hours
r0409  score=0.176  Power BI Data Analyst Associate Prep (3 h, intermediate)
        Teaches SQL (priority 0.72), Data visualisation (priority 0.16) in 3 hours
r0153  score=0.144  MySQL for Data Analytics and Business Intelligence (3 h, beginner)
        Teaches SQL (priority 0.72) in 3 hours
r0420  score=0.144  Database, Big Data, and DevOps Services in GCP (5 h, intermediate)
        Teaches SQL (priority 0.72) in 5 hours
r0104  score=0.144  Advanced Cybersecurity (5 h, intermediate)
        Teaches SQL (priority 0.72) in 5 hours
r0240  score=0.134  Build a Data Warehouse Using BigQuery (8 h, advanced)
        Teaches SQL (priority 0.72), Python (priority 0.35) in 8 hours
r0292  score=0.124  Modeling Climate Anomalies with Statistical Analysis (7 h, intermediate)
        Teaches Data visualisation (priority 0.16), Python (pri

In [ ]:
# ------------------------------------------------------------
# Difficulty filter: hide "advanced" courses when the learner's
# average mastery on the skills the course teaches is below 0.5
# ------------------------------------------------------------
import json

def filter_by_difficulty(scores, courses_df, learner_estimates, min_mastery=0.5):
    catalog = courses_df.set_index("course_id")

    def ready(course_id):
        if catalog.loc[course_id, "difficulty"] != "advanced":
            return True
        skills = catalog.loc[course_id, "esco_skill_ids"].split("|")
        avg = sum(learner_estimates.get(s, 0.0) for s in skills) / len(skills)
        return avg >= min_mastery

    return scores[scores["course_id"].apply(ready)]


def recommend_courses(job_skills, learner_estimates, course_catalogue,
                      completed=None, top_n=5):
    gaps = compute_skill_gaps(job_skills, learner_estimates)
    gaps_by_id = {g["esco_id"]: g for g in gaps}

    scores = score_courses(gaps, course_catalogue)
    available = filter_by_prerequisites(scores, course_catalogue, completed)
    available = filter_by_difficulty(available, course_catalogue, learner_estimates)
    top = available.head(top_n)

    skills_of = dict(zip(course_catalogue["course_id"],
                         course_catalogue["esco_skill_ids"]))
    results = []
    for _, row in top.iterrows():
        parts = [f"{gaps_by_id[s]['label']} (priority {gaps_by_id[s]['priority']})"
                 for s in skills_of[row["course_id"]].split("|")
                 if s in gaps_by_id and gaps_by_id[s]["priority"] > 0]
        results.append({
            "course_id": row["course_id"],
            "score": float(row["score"]),
            "reason": f"Teaches {', '.join(parts)} in {int(row['hours'])} hours",
        })
    return results


recs_real = recommend_courses(job_skills, learner_estimates, real_catalogue, top_n=10)

with open("/kaggle/working/data/mock/recommendations_real.json", "w") as f:
    json.dump(recs_real, f, indent=2)

for r in recs_real:
    row = info.loc[r["course_id"]]
    print(f"{r['course_id']}  score={r['score']:.3f}  "
          f"{row['title']} ({row['estimated_hours']:.0f} h, {row['difficulty']})")

r0263  score=0.180  The Nature of Data and Relational Database Design (6 h, beginner)
r0409  score=0.176  Power BI Data Analyst Associate Prep (3 h, intermediate)
r0153  score=0.144  MySQL for Data Analytics and Business Intelligence (3 h, beginner)
r0420  score=0.144  Database, Big Data, and DevOps Services in GCP (5 h, intermediate)
r0104  score=0.144  Advanced Cybersecurity (5 h, intermediate)
r0292  score=0.124  Modeling Climate Anomalies with Statistical Analysis (7 h, intermediate)
r0205  score=0.120  Vector Search with Relational Databases using PostgreSQL (6 h, intermediate)
r0124  score=0.120  Microsoft Azure SQL (6 h, beginner)
r0378  score=0.118  Data Science at Scale - Capstone Project (6 h, )
r0383  score=0.110  Advanced Data Analysis and Visualization with Pandas (5 h, advanced)


In [ ]:
# ------------------------------------------------------------
# Second learner: strong at SQL, weak at Python and statistics
# Same job, same catalogue, different learner
# ------------------------------------------------------------
bilal_estimates = {
    "skill_sql":    0.9,   # strong
    "skill_viz":    0.5,
    "skill_python": 0.1,   # very weak
    "skill_stats":  0.1,   # very weak
    "skill_excel":  0.8,
}

print("Gaps for Bilal:")
for g in compute_skill_gaps(job_skills, bilal_estimates):
    print(f"  {g['label']:20s} priority {g['priority']}")
print()

recs_bilal = recommend_courses(job_skills, bilal_estimates, real_catalogue, top_n=10)

print("Top 10 courses for Bilal:")
for r in recs_bilal:
    row = info.loc[r["course_id"]]
    print(f"{r['course_id']}  score={r['score']:.3f}  "
          f"{row['title']} ({row['estimated_hours']:.0f} h, {row['difficulty']})")

Gaps for Bilal:
  Python               priority 0.63
  Statistics           priority 0.54
  Data visualisation   priority 0.4
  SQL                  priority 0.09
  Excel                priority 0.08

Top 10 courses for Bilal:
r0292  score=0.224  Modeling Climate Anomalies with Statistical Analysis (7 h, intermediate)
r0378  score=0.195  Data Science at Scale - Capstone Project (6 h, )
r0334  score=0.188  From Data to Decisions: Finding Patterns with AI (4 h, beginner)
r0255  score=0.188  Data Science for Professionals (1 h, beginner)
r0164  score=0.131  Introduction to Bayesian Statistics (12 h, beginner)
r0390  score=0.126  Python for Active Defense (4 h, intermediate)
r0291  score=0.126  The Fundamentals of Computing Capstone Exam (1 h, intermediate)
r0068  score=0.126  HPE Aruba Networking Automation Basics (2 h, beginner)
r0196  score=0.126  Learn Python: 1 (1 h, beginner)
r0208  score=0.126  DevOps for Network Automation (NetDevOps) (4 h, intermediate)


In [ ]:
# ------------------------------------------------------------
# Look at the popularity columns for the courses in our catalogue
# (real = the 437 courses we kept in Step 14)
# ------------------------------------------------------------
print(real[["title", "enrolled", "rating", "num_reviews"]].head(10).to_string())
print()
print("Missing values:")
print(real[["enrolled", "rating", "num_reviews"]].isna().sum())

                                                             title                     enrolled            rating  num_reviews
0                             How to Create and Explore a Data Set  Enrollment number not found  Rating not found          NaN
1                  Meta Database Engineer Professional Certificate                       58,470               4.6       1977.0
2             Introduction to Machine Learning in Sports Analytics                        4,101               4.7         22.0
3                                           Data Science Companion  Enrollment number not found  Rating not found          NaN
4                         Python Programming for Quantum Computing  Enrollment number not found  Rating not found          NaN
5                     Data Visualization and Transformation with R  Enrollment number not found  Rating not found          NaN
6                       Business Intelligence and Visual Analytics                        3,526               4

In [ ]:
# ------------------------------------------------------------
# Popularity baseline: rank courses by number of enrolled learners
#   "58,470"                      -> 58470
#   "Enrollment number not found" -> blank (treated as 0 for ranking)
# Ignores the job and the learner on purpose.
# ------------------------------------------------------------
import pandas as pd

real["enrolled_num"] = pd.to_numeric(
    real["enrolled"].astype(str).str.replace(",", "", regex=False),
    errors="coerce")
real["rating_num"] = pd.to_numeric(real["rating"], errors="coerce")

print("Courses with known enrollment:", real["enrolled_num"].notna().sum(), "of", len(real))
print("Courses with known rating:    ", real["rating_num"].notna().sum(), "of", len(real))
print()

popularity = pd.DataFrame({
    "course_id":  courses_real["course_id"],
    "title":      courses_real["title"],
    "enrolled":   real["enrolled_num"].fillna(0),
    "rating":     real["rating_num"],
}).sort_values("enrolled", ascending=False)

print("Top 10 by popularity:")
print(popularity.head(10).to_string(index=False))

Courses with known enrollment: 290 of 437
Courses with known rating:     314 of 437

Top 10 by popularity:
course_id                                                     title  enrolled  rating
    r0195                       Python for Everybody Specialization 1702775.0     4.8
    r0181 Google IT Automation with Python Professional Certificate  735050.0     4.8
    r0187                  Excel Skills for Business Specialization  560711.0     4.9
    r0088           Applied Data Science with Python Specialization  425325.0     4.5
    r0250       Machine Learning Foundations: A Case Study Approach  394797.0     4.6
    r0162                          Mastering Data Analysis in Excel  345543.0     4.2
    r0125                 Introduction to Data Analysis Using Excel  275479.0     4.7
    r0301          Learn SQL Basics for Data Science Specialization  265453.0     4.5
    r0433                       Python 3 Programming Specialization  243225.0     4.8
    r0141                        

In [ ]:
# ------------------------------------------------------------
# Demand-only baseline: Score(c) = sum(importance) / max(hours, MIN_HOURS)
# Looks at the job, but ignores what the learner already knows
# ------------------------------------------------------------
def recommend_demand_only(job_skills, course_catalogue, top_n=5):
    importance = {s["esco_id"]: s["importance"] for s in job_skills}
    rows = []
    for _, c in course_catalogue.iterrows():
        skills = c["esco_skill_ids"].split("|")
        total = sum(importance.get(s, 0.0) for s in skills)
        rows.append({
            "course_id": c["course_id"],
            "score": total / max(c["estimated_hours"], MIN_HOURS),
        })
    out = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_n)
    return out["course_id"].tolist()

# Top 5 from each recommender, for Ayesha
pop_top    = popularity.head(5)["course_id"].tolist()
demand_top = recommend_demand_only(job_skills, real_catalogue, top_n=5)
pers_top   = [r["course_id"] for r in
              recommend_courses(job_skills, learner_estimates, real_catalogue, top_n=5)]

titles = real_catalogue.set_index("course_id")["title"]
skills_of = real_catalogue.set_index("course_id")["esco_skill_ids"]

for name, ids in [("POPULARITY", pop_top),
                  ("DEMAND-ONLY", demand_top),
                  ("PERSONALIZED", pers_top)]:
    print(name)
    for cid in ids:
        print(f"  {cid}  {titles[cid][:55]:55s} {skills_of[cid]}")
    print()

POPULARITY
  r0195  Python for Everybody Specialization                     skill_python
  r0181  Google IT Automation with Python Professional Certifica skill_python
  r0187  Excel Skills for Business Specialization                skill_excel
  r0088  Applied Data Science with Python Specialization         skill_python
  r0250  Machine Learning Foundations: A Case Study Approach     skill_python

DEMAND-ONLY
  r0383  Advanced Data Analysis and Visualization with Pandas    skill_viz|skill_python|skill_excel
  r0409  Power BI Data Analyst Associate Prep                    skill_sql|skill_viz
  r0292  Modeling Climate Anomalies with Statistical Analysis    skill_viz|skill_python|skill_stats
  r0334  From Data to Decisions: Finding Patterns with AI        skill_viz|skill_stats
  r0255  Data Science for Professionals                          skill_viz|skill_stats

PERSONALIZED
  r0263  The Nature of Data and Relational Database Design       skill_sql|skill_stats
  r0409  Power BI Data Anal

In [25]:
# ------------------------------------------------------------
# Rebuild Steps 27-31 in one cell:
# embedding model -> similarities -> hybrid tagging -> courses_real_v2.csv
# ------------------------------------------------------------
!pip install -q sentence-transformers

import re
import pandas as pd
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

targets = {
    "skill_sql":    "SQL",
    "skill_viz":    "data visualization",
    "skill_python": "Python",
    "skill_stats":  "statistics",
    "skill_excel":  "Excel",
}
rules = {
    "skill_sql":    r"\bsql\b|mysql|postgresql",
    "skill_viz":    r"visuali",
    "skill_python": r"\bpython\b",
    "skill_stats":  r"statistic",
    "skill_excel":  r"\bexcel\b",
}
thresholds = {"skill_sql": 0.60, "skill_excel": 0.60,
              "skill_python": 0.65, "skill_viz": 0.70, "skill_stats": 0.70}
sid_list = list(targets.keys())

unique_skills = sorted({s for lst in clean_df["skill_list"] for s in lst})
skill_vecs  = model.encode(unique_skills, convert_to_tensor=True, batch_size=128)
target_vecs = model.encode(list(targets.values()), convert_to_tensor=True)
sim_matrix  = util.cos_sim(target_vecs, skill_vecs).cpu().numpy()

skill_tags = {}
for j, text in enumerate(unique_skills):
    tags = set()
    for i, sid in enumerate(sid_list):
        if re.search(rules[sid], text) or sim_matrix[i][j] >= thresholds[sid]:
            tags.add(sid)
    skill_tags[text] = tags

def tag_skills_v2(skill_list):
    found = set().union(*[skill_tags[s] for s in skill_list])
    return "|".join(sid for sid in sid_list if sid in found)

clean_df["esco_skill_ids_v2"] = clean_df["skill_list"].apply(tag_skills_v2)
real2 = clean_df[clean_df["esco_skill_ids_v2"] != ""].reset_index(drop=True)

courses_real2 = pd.DataFrame({
    "course_id":       [f"r{i:04d}" for i in range(len(real2))],
    "title":           real2["title"],
    "description":     real2["Description"].fillna("").str.slice(0, 200),
    "esco_skill_ids":  real2["esco_skill_ids_v2"],
    "difficulty":      real2["Level"].str.replace(" level", "", regex=False).str.lower(),
    "estimated_hours": real2["estimated_hours"],
    "prerequisites":   "",
    "resource_url":    real2["URL"].fillna(""),
})
courses_real2.to_csv("/kaggle/working/data/mock/courses_real_v2.csv", index=False)

print("Courses saved:", len(courses_real2))
for sid in sid_list:
    print(f"  {sid:14s}", courses_real2["esco_skill_ids"].str.contains(sid).sum())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Courses saved: 450
  skill_sql      82
  skill_viz      128
  skill_python   160
  skill_stats    112
  skill_excel    45


In [26]:
# ------------------------------------------------------------
# Self-contained fix: define everything the comparison needs,
# then re-run the hybrid-v2 comparison (Experiment C)
# ------------------------------------------------------------
import pandas as pd

# 1) The two learners
bilal_estimates = {
    "skill_sql": 0.9, "skill_viz": 0.5, "skill_python": 0.1,
    "skill_stats": 0.1, "skill_excel": 0.8,
}
learners = {"Ayesha": learner_estimates, "Bilal": bilal_estimates}

# 2) Gap coverage metric
def gap_coverage(course_ids, gaps, catalogue):
    skills_of = catalogue.set_index("course_id")["esco_skill_ids"]
    covered = {s for cid in course_ids for s in skills_of[cid].split("|")}
    priority = {g["esco_id"]: g["priority"] for g in gaps}
    return sum(p for s, p in priority.items() if s in covered) / sum(priority.values())

# 3) Demand-only baseline
def recommend_demand_only(job_skills, course_catalogue, top_n=5):
    importance = {s["esco_id"]: s["importance"] for s in job_skills}
    rows = []
    for _, c in course_catalogue.iterrows():
        total = sum(importance.get(s, 0.0) for s in c["esco_skill_ids"].split("|"))
        rows.append({"course_id": c["course_id"],
                     "score": total / max(c["estimated_hours"], MIN_HOURS)})
    out = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_n)
    return out["course_id"].tolist()

# 4) Diverse picker
def recommend_courses_diverse(job_skills, learner_estimates, catalogue,
                              completed=None, top_n=5):
    remaining = [dict(g) for g in compute_skill_gaps(job_skills, learner_estimates)]
    skills_of = catalogue.set_index("course_id")["esco_skill_ids"]
    chosen = []
    for _ in range(top_n):
        scores = score_courses(remaining, catalogue)
        avail = filter_by_prerequisites(scores, catalogue, completed)
        avail = filter_by_difficulty(avail, catalogue, learner_estimates)
        avail = avail[~avail["course_id"].isin(chosen) & (avail["score"] > 0)]
        if avail.empty:
            break
        best = avail.iloc[0]["course_id"]
        chosen.append(best)
        covered = set(skills_of[best].split("|"))
        for g in remaining:
            if g["esco_id"] in covered:
                g["priority"] = 0.0
    return chosen

# 5) The comparison on the hybrid-v2 catalogue
real_catalogue2 = pd.read_csv("/kaggle/working/data/mock/courses_real_v2.csv",
                              keep_default_na=False)
pop2 = pd.DataFrame({
    "course_id": real_catalogue2["course_id"],
    "enrolled": pd.to_numeric(
        real2["enrolled"].astype(str).str.replace(",", "", regex=False),
        errors="coerce").fillna(0).values,
}).sort_values("enrolled", ascending=False)

pop_top2    = pop2.head(5)["course_id"].tolist()
demand_top2 = recommend_demand_only(job_skills, real_catalogue2, top_n=5)
hours_of2   = real_catalogue2.set_index("course_id")["estimated_hours"]

print(f"{'Learner':8s} {'Method':14s} {'Coverage':>9s} {'Hours':>7s}")
for name, est in learners.items():
    gaps_l = compute_skill_gaps(job_skills, est)
    methods = {
        "Popularity":  pop_top2,
        "Demand-only": demand_top2,
        "Diverse":     recommend_courses_diverse(job_skills, est, real_catalogue2),
    }
    for m, ids in methods.items():
        print(f"{name:8s} {m:14s} {gap_coverage(ids, gaps_l, real_catalogue2):9.2f} "
              f"{sum(hours_of2[c] for c in ids):7.0f}")
    print()

Learner  Method          Coverage   Hours
Ayesha   Popularity          0.24     618
Ayesha   Demand-only         1.00      20
Ayesha   Diverse             1.00      11

Bilal    Popularity          0.41     618
Bilal    Demand-only         1.00      20
Bilal    Diverse             1.00      10



In [27]:
# ------------------------------------------------------------
# Diverse recommender: pick one course at a time.
# After each pick, set the priority of the skills it teaches to 0,
# so the next course must close a DIFFERENT gap.
# ------------------------------------------------------------
def recommend_courses_diverse(job_skills, learner_estimates, catalogue,
                              completed=None, top_n=5):
    remaining = [dict(g) for g in compute_skill_gaps(job_skills, learner_estimates)]
    skills_of = catalogue.set_index("course_id")["esco_skill_ids"]
    chosen = []

    for _ in range(top_n):
        scores = score_courses(remaining, catalogue)
        avail = filter_by_prerequisites(scores, catalogue, completed)
        avail = filter_by_difficulty(avail, catalogue, learner_estimates)
        avail = avail[~avail["course_id"].isin(chosen) & (avail["score"] > 0)]
        if avail.empty:
            break
        best = avail.iloc[0]["course_id"]
        chosen.append(best)
        covered = set(skills_of[best].split("|"))
        for g in remaining:
            if g["esco_id"] in covered:
                g["priority"] = 0.0
    return chosen

hours_of = real_catalogue.set_index("course_id")["estimated_hours"]

print(f"{'Learner':8s} {'Method':14s} {'Coverage':>9s} {'Hours':>7s}")
for name, est in learners.items():
    gaps_l = compute_skill_gaps(job_skills, est)
    methods = {
        "Popularity":   pop_top,
        "Demand-only":  demand_top,
        "Personalized": [r["course_id"] for r in
                         recommend_courses(job_skills, est, real_catalogue, top_n=5)],
        "Diverse":      recommend_courses_diverse(job_skills, est, real_catalogue),
    }
    for m, ids in methods.items():
        cov = gap_coverage(ids, gaps_l, real_catalogue)
        hrs = sum(hours_of[c] for c in ids)
        print(f"{name:8s} {m:14s} {cov:9.2f} {hrs:7.0f}")
    print()

Learner  Method          Coverage   Hours
Ayesha   Popularity          0.24     618
Ayesha   Demand-only         1.00      20
Ayesha   Personalized        0.76      22
Ayesha   Diverse             1.00      11

Bilal    Popularity          0.41     618
Bilal    Demand-only         1.00      20
Bilal    Personalized        0.90      30
Bilal    Diverse             1.00      15



In [29]:
# ------------------------------------------------------------
# Final recommender: diverse picking, team-contract output
#   Output: [{course_id, score, reason}]
#   score  = NEW gap closed (not covered by earlier picks) per hour
# ------------------------------------------------------------
import json

def recommend_courses_final(job_skills, learner_estimates, catalogue,
                            completed=None, top_n=5):
    ids = recommend_courses_diverse(job_skills, learner_estimates, catalogue,
                                    completed, top_n)
    gaps_by_id = {g["esco_id"]: g
                  for g in compute_skill_gaps(job_skills, learner_estimates)}
    cat = catalogue.set_index("course_id")

    results, covered = [], set()
    for cid in ids:
        skills = cat.loc[cid, "esco_skill_ids"].split("|")
        new = [s for s in skills
               if s in gaps_by_id and gaps_by_id[s]["priority"] > 0 and s not in covered]
        covered.update(skills)
        hours = cat.loc[cid, "estimated_hours"]
        score = sum(gaps_by_id[s]["priority"] for s in new) / max(hours, MIN_HOURS)
        parts = ", ".join(f"{gaps_by_id[s]['label']} (priority {gaps_by_id[s]['priority']})"
                          for s in new)
        results.append({
            "course_id": cid,
            "score": round(float(score), 4),
            "reason": f"Closes new gap: {parts} in {int(hours)} hours",
        })
    return results

final_recs = recommend_courses_final(job_skills, learner_estimates, real_catalogue)

with open("/kaggle/working/data/mock/recommendations_real.json", "w") as f:
    json.dump(final_recs, f, indent=2)

titles = real_catalogue.set_index("course_id")["title"]
for r in final_recs:
    print(f"{r['course_id']}  score={r['score']:.3f}  {titles[r['course_id']]}")
    print(f"        {r['reason']}")

r0263  score=0.180  The Nature of Data and Relational Database Design
        Closes new gap: SQL (priority 0.72), Statistics (priority 0.36) in 6 hours
r0383  score=0.110  Advanced Data Analysis and Visualization with Pandas
        Closes new gap: Data visualisation (priority 0.16), Python (priority 0.35), Excel (priority 0.04) in 5 hours


In [30]:
# ------------------------------------------------------------
# 1) The agreed team name now points to the final version
# 2) Sanity test on the MOCK catalogue (Step 4), where we know
#    the expected answer from the project plan's worked example
# ------------------------------------------------------------
recommend_courses = recommend_courses_final

mock_recs = recommend_courses(job_skills, learner_estimates, courses_df)

mock_titles = courses_df.set_index("course_id")["title"]
for r in mock_recs:
    print(f"{r['course_id']}  score={r['score']:.3f}  {mock_titles[r['course_id']]}")

# Automatic checks
ids = [r["course_id"] for r in mock_recs]
assert ids[0] == "c01", "SQL Basics should rank first"
assert "c02" not in ids, "Advanced SQL needs c01 first, so it must be hidden"
assert all(set(r) == {"course_id", "score", "reason"} for r in mock_recs), "wrong output shape"
print("\nAll 3 checks passed")

c01  score=0.120  SQL Basics
c03  score=0.035  Python for Beginners
c05  score=0.026  Statistics Foundations
c04  score=0.020  Data Visualisation 101
c06  score=0.008  Excel Essentials

All 3 checks passed


In [31]:
# ------------------------------------------------------------
# Write the final functions into the agreed src/ files
# ------------------------------------------------------------
import os
os.makedirs("/kaggle/working/src", exist_ok=True)

skill_gaps_code = '''"""Skill gap: Priority(s) = importance * (1 - mastery)."""

def compute_skill_gaps(job_skills, learner_estimates):
    gaps = []
    for s in job_skills:
        d = s["importance"]
        m = learner_estimates.get(s["esco_id"], 0.0)  # unknown skill -> 0
        gaps.append({
            "esco_id": s["esco_id"], "label": s["label"],
            "importance": d, "mastery": m,
            "priority": round(d * (1 - m), 3),
        })
    return sorted(gaps, key=lambda g: g["priority"], reverse=True)
'''

recommendations_code = '''"""Course ranking: gap-based score, prerequisites, difficulty, diversity."""
import pandas as pd
from skill_gaps import compute_skill_gaps

MIN_HOURS = 5  # stops 1-2 hour courses from winning just by being short


def score_courses(gaps, courses_df, min_hours=MIN_HOURS):
    priority = {g["esco_id"]: g["priority"] for g in gaps}
    rows = []
    for _, c in courses_df.iterrows():
        total = sum(priority.get(s, 0.0) for s in c["esco_skill_ids"].split("|"))
        rows.append({
            "course_id": c["course_id"], "title": c["title"],
            "hours": c["estimated_hours"], "gap_closed": round(total, 3),
            "score": round(total / max(c["estimated_hours"], min_hours), 4),
        })
    return pd.DataFrame(rows).sort_values("score", ascending=False)


def filter_by_prerequisites(scores, courses_df, completed=None):
    completed = completed or set()
    prereq = dict(zip(courses_df["course_id"], courses_df["prerequisites"]))
    def unlocked(cid):
        return all(p in completed for p in prereq[cid].split("|") if p)
    return scores[scores["course_id"].apply(unlocked)]


def filter_by_difficulty(scores, courses_df, learner_estimates, min_mastery=0.5):
    catalog = courses_df.set_index("course_id")
    def ready(cid):
        if catalog.loc[cid, "difficulty"] != "advanced":
            return True
        skills = catalog.loc[cid, "esco_skill_ids"].split("|")
        return sum(learner_estimates.get(s, 0.0) for s in skills) / len(skills) >= min_mastery
    return scores[scores["course_id"].apply(ready)]


def _pick_diverse(job_skills, learner_estimates, catalogue, completed, top_n):
    remaining = [dict(g) for g in compute_skill_gaps(job_skills, learner_estimates)]
    skills_of = catalogue.set_index("course_id")["esco_skill_ids"]
    chosen = []
    for _ in range(top_n):
        scores = score_courses(remaining, catalogue)
        avail = filter_by_prerequisites(scores, catalogue, completed)
        avail = filter_by_difficulty(avail, catalogue, learner_estimates)
        avail = avail[~avail["course_id"].isin(chosen) & (avail["score"] > 0)]
        if avail.empty:
            break
        best = avail.iloc[0]["course_id"]
        chosen.append(best)
        covered = set(skills_of[best].split("|"))
        for g in remaining:
            if g["esco_id"] in covered:
                g["priority"] = 0.0
    return chosen


def recommend_courses(job_skills, learner_estimates, course_catalogue,
                      completed=None, top_n=5):
    """Team contract: returns [{course_id, score, reason}]."""
    ids = _pick_diverse(job_skills, learner_estimates, course_catalogue, completed, top_n)
    gaps_by_id = {g["esco_id"]: g for g in compute_skill_gaps(job_skills, learner_estimates)}
    cat = course_catalogue.set_index("course_id")
    results, covered = [], set()
    for cid in ids:
        skills = cat.loc[cid, "esco_skill_ids"].split("|")
        new = [s for s in skills
               if s in gaps_by_id and gaps_by_id[s]["priority"] > 0 and s not in covered]
        covered.update(skills)
        hours = cat.loc[cid, "estimated_hours"]
        score = sum(gaps_by_id[s]["priority"] for s in new) / max(hours, MIN_HOURS)
        parts = ", ".join(f"{gaps_by_id[s]['label']} (priority {gaps_by_id[s]['priority']})"
                          for s in new)
        results.append({"course_id": cid, "score": round(float(score), 4),
                        "reason": f"Closes new gap: {parts} in {int(hours)} hours"})
    return results
'''

with open("/kaggle/working/src/skill_gaps.py", "w") as f:
    f.write(skill_gaps_code)
with open("/kaggle/working/src/recommendations.py", "w") as f:
    f.write(recommendations_code)

# Test: import the FILES (not the notebook cells) and run on the mock data
import sys, importlib
sys.path.insert(0, "/kaggle/working/src")
import skill_gaps, recommendations
importlib.reload(skill_gaps); importlib.reload(recommendations)

test = recommendations.recommend_courses(job_skills, learner_estimates, courses_df)
print([r["course_id"] for r in test])
assert [r["course_id"] for r in test] == [r["course_id"] for r in mock_recs]
print("src files work and match the notebook results")

['c01', 'c03', 'c05', 'c04', 'c06']
src files work and match the notebook results


In [32]:
# ------------------------------------------------------------
# Load a free, open-source sentence-embedding model and check
# how similar some skill phrases are to our job skill "SQL".
# Similarity: 1.0 = same meaning, 0 = unrelated
# ------------------------------------------------------------
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

target = "SQL"
candidates = [
    "mysql", "postgresql", "relational databases", "database design",
    "sql injection", "cybersecurity", "python", "data visualization",
    "cooking", "web interactivity",
]

target_vec = model.encode(target, convert_to_tensor=True)
cand_vecs  = model.encode(candidates, convert_to_tensor=True)
sims = util.cos_sim(target_vec, cand_vecs)[0]

for c, s in sorted(zip(candidates, sims.tolist()), key=lambda x: -x[1]):
    print(f"{c:22s} {s:.2f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

mysql                  0.69
sql injection          0.65
relational databases   0.63
database design        0.56
postgresql             0.51
python                 0.30
data visualization     0.29
cybersecurity          0.26
cooking                0.25
web interactivity      0.24


In [33]:
# ------------------------------------------------------------
# Compare every unique course skill with our 5 job skills.
# Print how many match at threshold 0.45, plus borderline cases
# (similarity 0.40 - 0.55) so we can judge them by eye.
# ------------------------------------------------------------
import numpy as np

targets = {
    "skill_sql":    "SQL",
    "skill_viz":    "data visualization",
    "skill_python": "Python",
    "skill_stats":  "statistics",
    "skill_excel":  "Excel",
}
THRESHOLD = 0.45

# every distinct skill text in the cleaned catalogue
unique_skills = sorted({s for lst in clean_df["skill_list"] for s in lst})
print("Unique course skills:", len(unique_skills))

skill_vecs  = model.encode(unique_skills, convert_to_tensor=True, batch_size=128)
target_vecs = model.encode(list(targets.values()), convert_to_tensor=True)
sim_matrix  = util.cos_sim(target_vecs, skill_vecs).cpu().numpy()   # 5 x N

for i, (sid, name) in enumerate(targets.items()):
    sims = sim_matrix[i]
    matched = (sims >= THRESHOLD).sum()
    print(f"\n=== {sid} ({name}):  {matched} skills match at >= {THRESHOLD} ===")

    border = [(unique_skills[j], sims[j]) for j in range(len(unique_skills))
              if 0.40 <= sims[j] <= 0.55]
    border.sort(key=lambda x: -x[1])
    for text, s in border[:12]:
        verdict = "MATCH" if s >= THRESHOLD else "no   "
        print(f"   {s:.2f} {verdict}  {text}")

Unique course skills: 7500

=== skill_sql (SQL):  82 skills match at >= 0.45 ===
   0.54 MATCH  table operations
   0.54 MATCH  database administration
   0.54 MATCH  using subqueries
   0.54 MATCH  ensures reliable sql operations.
   0.54 MATCH  spark sql
   0.52 MATCH  data analytics
   0.52 MATCH  database (db) design
   0.52 MATCH  database (dbsm)
   0.51 MATCH  postgresql
   0.50 MATCH  database management systems
   0.50 MATCH  relational database management systems
   0.49 MATCH  sql server 2024

=== skill_viz (data visualization):  145 skills match at >= 0.45 ===
   0.55 MATCH  visualizing query results in databricks
   0.55 MATCH  data analysis tools
   0.54 MATCH  data engineering
   0.54 MATCH  scatter plots
   0.54 MATCH  organizing data
   0.54 MATCH  illustrate complex queries for efficient data analysis
   0.54 MATCH  infographics
   0.54 MATCH  data reporting
   0.54 MATCH  data analysis software
   0.53 MATCH  data storytelling
   0.53 MATCH  data set analysis
   0.53 

In [34]:
# ------------------------------------------------------------
# How many skills match at different thresholds, and what do
# the matches look like just above 0.45?
# ------------------------------------------------------------
import random
random.seed(0)

print(f"{'skill':14s} " + " ".join(f"{t:>6.2f}" for t in [0.45, 0.50, 0.55, 0.60, 0.65]))
for i, sid in enumerate(targets):
    sims = sim_matrix[i]
    counts = [(sims >= t).sum() for t in [0.45, 0.50, 0.55, 0.60, 0.65]]
    print(f"{sid:14s} " + " ".join(f"{c:6d}" for c in counts))

print("\nRandom sample of matches between 0.55 and 0.70:")
for i, sid in enumerate(targets):
    sims = sim_matrix[i]
    band = [(unique_skills[j], sims[j]) for j in range(len(unique_skills))
            if 0.55 <= sims[j] < 0.70]
    sample = random.sample(band, min(8, len(band)))
    print(f"\n=== {sid}  ({len(band)} skills in this band) ===")
    for text, s in sorted(sample, key=lambda x: -x[1]):
        print(f"   {s:.2f}  {text}")

skill            0.45   0.50   0.55   0.60   0.65
skill_sql          82     53     43     25     16
skill_viz         145     85     53     35     25
skill_python       69     38     28     16      9
skill_stats        87     60     38     26     14
skill_excel        48     24     19     16     15

Random sample of matches between 0.55 and 0.70:

=== skill_sql  (34 skills in this band) ===
   0.66  sql & rdbms
   0.61  databases
   0.60  sql database management
   0.60  sql and python syntax
   0.59  data and databases
   0.58  quering databases
   0.57  database comparison
   0.57  oracle databases

=== skill_viz  (32 skills in this band) ===
   0.68  data exploration
   0.64  data presentations
   0.64  data wrangling
   0.62  creating charts
   0.62  data analysis
   0.59  visualize complex data trapped in pdfs
   0.59  data-analysis
   0.55  data modeling

=== skill_python  (22 skills in this band) ===
   0.70  python libraries
   0.62  computer programming
   0.61  programming
  

In [36]:
# ------------------------------------------------------------
# Hybrid tagger:  keyword rule  OR  embedding similarity >= 0.60
# Saves courses_real_v2.csv (old courses_real.csv is untouched)
# ------------------------------------------------------------
EMB_THRESHOLD = 0.60
sid_list = list(targets.keys())

skill_tags, emb_only = {}, {sid: [] for sid in sid_list}
for j, text in enumerate(unique_skills):
    tags = set()
    for i, sid in enumerate(sid_list):
        by_keyword = re.search(rules[sid], text) is not None
        by_embed   = sim_matrix[i][j] >= EMB_THRESHOLD
        if by_keyword or by_embed:
            tags.add(sid)
        if by_embed and not by_keyword:
            emb_only[sid].append((text, float(sim_matrix[i][j])))
    skill_tags[text] = tags

def tag_skills_v2(skill_list):
    found = set().union(*[skill_tags[s] for s in skill_list])
    return "|".join(sid for sid in sid_list if sid in found)

clean_df["esco_skill_ids_v2"] = clean_df["skill_list"].apply(tag_skills_v2)
real2 = clean_df[clean_df["esco_skill_ids_v2"] != ""].reset_index(drop=True)

courses_real2 = pd.DataFrame({
    "course_id":       [f"r{i:04d}" for i in range(len(real2))],
    "title":           real2["title"],
    "description":     real2["Description"].fillna("").str.slice(0, 200),
    "esco_skill_ids":  real2["esco_skill_ids_v2"],
    "difficulty":      real2["Level"].str.replace(" level", "", regex=False).str.lower(),
    "estimated_hours": real2["estimated_hours"],
    "prerequisites":   "",
    "resource_url":    real2["URL"].fillna(""),
})
courses_real2.to_csv("/kaggle/working/data/mock/courses_real_v2.csv", index=False)

print(f"Courses: {len(courses_real)} (keywords only) -> {len(courses_real2)} (hybrid)")
print("\nCourses per skill (keywords only -> hybrid):")
for sid in sid_list:
    old = courses_real["esco_skill_ids"].str.contains(sid).sum()
    new = courses_real2["esco_skill_ids"].str.contains(sid).sum()
    print(f"  {sid:14s} {old:4d} -> {new:4d}")

print("\nSkills added by the embedding ONLY (top 8 per skill):")
for sid in sid_list:
    print(f"\n  {sid}  ({len(emb_only[sid])} skills added)")
    for text, s in sorted(emb_only[sid], key=lambda x: -x[1])[:8]:
        print(f"     {s:.2f}  {text}")

Courses: 437 (keywords only) -> 603 (hybrid)

Courses per skill (keywords only -> hybrid):
  skill_sql        72 ->   82
  skill_viz       127 ->  274
  skill_python    160 ->  212
  skill_stats     112 ->  152
  skill_excel      39 ->   45

Skills added by the embedding ONLY (top 8 per skill):

  skill_sql  (10 skills added)
     0.67  relational database
     0.67  data queries
     0.67  data querying
     0.63  relational databases
     0.63  table joins
     0.61  database basics
     0.61  databases
     0.61  database application

  skill_viz  (13 skills added)
     0.76  represent data visually
     0.72  visual analytics
     0.68  data exploration
     0.67  basic understanding of data analysis and charts
     0.64  exploratory data analysis
     0.64  data presentations
     0.64  data wrangling
     0.63  data exploration techniques

  skill_python  (4 skills added)
     0.63  scipy
     0.62  computer programming
     0.62  programming tool
     0.61  programming

  skill_

In [37]:
# ------------------------------------------------------------
# Hybrid tagger v2: keyword rule OR embedding >= per-skill threshold
# ------------------------------------------------------------
thresholds = {
    "skill_sql":    0.60,
    "skill_excel":  0.60,
    "skill_python": 0.65,
    "skill_viz":    0.70,
    "skill_stats":  0.70,
}

skill_tags, emb_only = {}, {sid: [] for sid in sid_list}
for j, text in enumerate(unique_skills):
    tags = set()
    for i, sid in enumerate(sid_list):
        by_keyword = re.search(rules[sid], text) is not None
        by_embed   = sim_matrix[i][j] >= thresholds[sid]
        if by_keyword or by_embed:
            tags.add(sid)
        if by_embed and not by_keyword:
            emb_only[sid].append((text, float(sim_matrix[i][j])))
    skill_tags[text] = tags

clean_df["esco_skill_ids_v2"] = clean_df["skill_list"].apply(tag_skills_v2)
real2 = clean_df[clean_df["esco_skill_ids_v2"] != ""].reset_index(drop=True)

courses_real2 = pd.DataFrame({
    "course_id":       [f"r{i:04d}" for i in range(len(real2))],
    "title":           real2["title"],
    "description":     real2["Description"].fillna("").str.slice(0, 200),
    "esco_skill_ids":  real2["esco_skill_ids_v2"],
    "difficulty":      real2["Level"].str.replace(" level", "", regex=False).str.lower(),
    "estimated_hours": real2["estimated_hours"],
    "prerequisites":   "",
    "resource_url":    real2["URL"].fillna(""),
})
courses_real2.to_csv("/kaggle/working/data/mock/courses_real_v2.csv", index=False)

print(f"Courses: {len(courses_real)} (keywords only) -> {len(courses_real2)} (hybrid v2)")
print("\nCourses per skill (keywords only -> hybrid v2):")
for sid in sid_list:
    old = courses_real["esco_skill_ids"].str.contains(sid).sum()
    new = courses_real2["esco_skill_ids"].str.contains(sid).sum()
    print(f"  {sid:14s} {old:4d} -> {new:4d}   (embedding added {len(emb_only[sid])} skills)")

Courses: 437 (keywords only) -> 450 (hybrid v2)

Courses per skill (keywords only -> hybrid v2):
  skill_sql        72 ->   82   (embedding added 10 skills)
  skill_viz       127 ->  128   (embedding added 2 skills)
  skill_python    160 ->  160   (embedding added 0 skills)
  skill_stats     112 ->  112   (embedding added 0 skills)
  skill_excel      39 ->   45   (embedding added 5 skills)


In [38]:
# ------------------------------------------------------------
# Re-run Experiment C on the hybrid-v2 catalogue (450 courses)
# Methods: Popularity, Demand-only, Diverse (our final recommender)
# ------------------------------------------------------------
real_catalogue2 = pd.read_csv("/kaggle/working/data/mock/courses_real_v2.csv",
                              keep_default_na=False)

# Popularity: most enrolled first (ignores job and learner)
pop2 = pd.DataFrame({
    "course_id": courses_real2["course_id"],
    "enrolled": pd.to_numeric(
        real2["enrolled"].astype(str).str.replace(",", "", regex=False),
        errors="coerce").fillna(0),
}).sort_values("enrolled", ascending=False)

pop_top2    = pop2.head(5)["course_id"].tolist()
demand_top2 = recommend_demand_only(job_skills, real_catalogue2, top_n=5)
hours_of2   = real_catalogue2.set_index("course_id")["estimated_hours"]
titles2     = real_catalogue2.set_index("course_id")["title"]

print(f"{'Learner':8s} {'Method':14s} {'Coverage':>9s} {'Hours':>7s}")
for name, est in learners.items():
    gaps_l = compute_skill_gaps(job_skills, est)
    methods = {
        "Popularity":  pop_top2,
        "Demand-only": demand_top2,
        "Diverse":     recommend_courses_diverse(job_skills, est, real_catalogue2),
    }
    for m, ids in methods.items():
        cov = gap_coverage(ids, gaps_l, real_catalogue2)
        hrs = sum(hours_of2[c] for c in ids)
        print(f"{name:8s} {m:14s} {cov:9.2f} {hrs:7.0f}")
    print()

print("Final recommendations for Ayesha (hybrid v2):")
for r in recommend_courses(job_skills, learner_estimates, real_catalogue2):
    print(f"  {r['course_id']}  {titles2[r['course_id']]}")
    print(f"        {r['reason']}")

Learner  Method          Coverage   Hours
Ayesha   Popularity          0.24     618
Ayesha   Demand-only         1.00      20
Ayesha   Diverse             1.00      11

Bilal    Popularity          0.41     618
Bilal    Demand-only         1.00      20
Bilal    Diverse             1.00      10

Final recommendations for Ayesha (hybrid v2):
  r0275  The Nature of Data and Relational Database Design
        Closes new gap: SQL (priority 0.72), Statistics (priority 0.36) in 6 hours
  r0396  Advanced Data Analysis and Visualization with Pandas
        Closes new gap: Data visualisation (priority 0.16), Python (priority 0.35), Excel (priority 0.04) in 5 hours


In [39]:
# ------------------------------------------------------------
# Pack everything Member 5 needs to commit into ONE zip file
# and show which files are inside
# ------------------------------------------------------------
import os, shutil

W = "/kaggle/working"
needed = ["src/skill_gaps.py", "src/recommendations.py",
          "data/mock/job_skills.json", "data/mock/learner_estimates.json",
          "data/mock/courses.csv", "data/mock/recommendations.json",
          "data/mock/courses_real_v2.csv", "data/mock/recommendations_real.json"]

for f in needed:
    print("OK     " if os.path.exists(f"{W}/{f}") else "MISSING", f)

# zip the three folders/files into one archive
os.makedirs(f"{W}/member5", exist_ok=True)
for item in ["src", "data", "LIMITATIONS.md"]:
    src = f"{W}/{item}"
    if os.path.exists(src):
        dst = f"{W}/member5/{item}"
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst) if os.path.isdir(src) else shutil.copy(src, dst)

shutil.make_archive(f"{W}/member5_deliverables", "zip", f"{W}/member5")
print("\nCreated member5_deliverables.zip")

OK      src/skill_gaps.py
OK      src/recommendations.py
OK      data/mock/job_skills.json
OK      data/mock/learner_estimates.json
OK      data/mock/courses.csv
OK      data/mock/recommendations.json
OK      data/mock/courses_real_v2.csv
OK      data/mock/recommendations_real.json

Created member5_deliverables.zip
